# 第8章　不均衡への実装的対処 ― サンプリングと難例マイニング

**『本格実装 医療診断支援AI（実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## 8.1　少数クラスを、多く見せる ― オーバーサンプリング

In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

labels = train_df["label"].values                    # 各症例のクラス
class_count = np.bincount(labels)
weight_per_class = 1.0 / class_count                 # 少ないクラスほど大きい重み
sample_weights = weight_per_class[labels]            # 各症例の重み
sampler = WeightedRandomSampler(sample_weights, num_samples=len(labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)  # shuffleの代わり

## 8.2　病変を含むパッチを、優先して切る

In [ ]:
from monai.transforms import RandCropByPosNegLabeld
crop = RandCropByPosNegLabeld(
    keys=["image", "label"], label_key="label",
    spatial_size=(96, 96, 96),
    pos=3, neg=1,                # 前景中心:背景中心の選択重み = 3:1（病変を含むパッチの比率ではない）
    num_samples=4,
)

## 8.3　難しい症例を、重点的に ― 難例マイニング

In [ ]:
# バッチ内で、損失の大きい上位を選んで学習する（OHEM の簡易版）
losses = criterion_none(model(x), y)          # reduction="none" で各サンプルの損失
if losses.dim() > 1:                          # セグメンテーションでは画素ごとに出るので、
    losses = losses.flatten(1).mean(1)        # まず症例内で平均し、症例単位の1次元にする
k = max(1, int(len(losses) * 0.7))             # 上位70%。バッチ1でも0本にならないようにする
hard = losses.topk(k).values.mean()
optimizer.zero_grad(set_to_none=True)
hard.backward()
optimizer.step()

## 損失関数の側から不均衡に対処する ― Tversky（トベルスキー）の β を調整する

In [ ]:
def tversky_loss(prob, target, beta=0.7, eps=1e-6):
    tp = (prob * target).sum()
    fp = (prob * (1 - target)).sum()
    fn = ((1 - prob) * target).sum()
    ti = (tp + eps) / (tp + (1 - beta) * fp + beta * fn + eps)
    return 1 - ti

## 易しい順に学ばせる ― カリキュラム学習

In [ ]:
# 難易度スコア（小さいほど易しい）で並べ、エポックとともに窓を広げる
def curriculum_indices(difficulty, epoch, warmup=20):
    order = difficulty.argsort()                 # 易しい順に整列
    frac = min(1.0, 0.3 + 0.7 * epoch / warmup)  # 3割から徐々に全体へ
    k = int(len(order) * frac)
    return order[:k]                             # 今エポックで使う症例集合

## 長尾分布を正面から扱う ― サンプリングの先へ

In [ ]:
def class_balanced_weights(class_count, beta=0.999):
    eff = (1.0 - np.power(beta, class_count)) / (1.0 - beta)  # 有効サンプル数
    w = 1.0 / eff
    return w / w.sum() * len(class_count)     # 平均1に正規化して損失へ渡す

In [ ]:
log_prior = torch.log(torch.tensor(class_count) / class_count.sum())
import torch.nn.functional as F

def logit_adjusted_ce(logits, y, tau=1.0):
    return F.cross_entropy(logits + tau * log_prior.to(logits.device), y)  # 推論時は素のlogits